# 教程 1：观测 —— 从一行导出到 SAE 特征落盘

**场景**：生成过程中，用稀疏自编码器（SAE）的编码器观察第 3、5、17 层的 hidden，
特征打印到控制台并写入 pkl。模型行为不许有任何改变。

我们不一步到位，而是分四步，每步只引入一个新概念：

1. 导出原始 hidden（复习教程 0，看清导出语义）；
2. 观测头需要参数 → **冻结状态**；
3. 三层一起观测 → **多输入任务**；
4. 生成的同时打印与落盘 → **trace 的并行消费**。

## 第一步：导出原始 hidden

教程 0 导出了行均值。这次导出原始 hidden 的若干行若干列，看清两个边界参数：

- `max_rows` / `max_cols` 是**硬边界**：输入帧和输出都被裁到这个尺寸，
  而且 `max_rows × max_cols × 8` 不能超过单个快照的 32 KiB；
- 导出的张量是**宿主机拥有的 CPU 副本**——stride 门控复制，不阻塞 CUDA 流。

In [1]:
import os
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")

from typing import Annotated
import torch
import pllm
from vllm import SamplingParams
from pllm.config import VllmDType, ModelConfig


class RawPeek(pllm.Workload):
    """每步导出第 3 层 hidden 的前 2 行、前 8 列。"""

    @pllm.on(pllm.Phase.OBSERVE)
    @pllm.export(name="raw", every=1, max_rows=2, max_cols=8)
    def observe(self, hidden: Annotated[torch.Tensor, pllm.Hidden(3)]) -> torch.Tensor:
        return hidden.float()   # 帧已被裁到 (≤2 行, 8 列)


llm = pllm.make_llm(workload=RawPeek(), config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", VllmDType.BFLOAT16, 64, 2, 128, 0.2, seed=7))
llm.generate(["The sky is", "Water freezes at"],
    SamplingParams(max_tokens=3, temperature=0.0, ignore_eos=True), use_tqdm=False)
for event in llm.trace_snapshot().events:
    if event.tensor_name == "raw":
        print(f"step {event.step_id}: shape={tuple(event.tensor().shape)}, "
              f"首行={event.tensor()[0, :4].tolist()}")

/workspace/pllm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-10 12:28:19,603	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 09-10 12:28:20 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 128, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 12:28:20 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 12:28:22 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 12:28:22 [model.py:1883] Using max model len 64


... (26 outputs left out)

INFO 09-10 12:28:29 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


INFO 09-10 12:28:29 [core.py:355] init engine (profile, create kv cache, warmup model) took 1.60 s


INFO 09-10 12:28:33 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


step 1: shape=(2, 8), 首行=[-0.0277099609375, -0.015869140625, -0.04541015625, -0.060302734375]
step 2: shape=(2, 8), 首行=[0.1435546875, -0.06591796875, -0.1396484375, -0.044921875]


## 第二步：观测头需要参数 —— 冻结状态

SAE 的编码器是一个矩阵 `W`（字典大小 × hidden 宽度）。它不是训练目标，
只是观测基础设施：初始化一次，之后只读。

PLLM 里这就是**冻结状态**：`pllm.InstanceState` 不给优化器。

- admission 时运行 `initialize`（这里是正交初始化，用框架给的 generator，结果可复现）；
- 任何任务 —— 包括观测任务 —— 可以用 `pllm.Parameter("encoder", "w3")` 读它；
- 它永远不会被更新：观测的签名约束是"不改变模型状态"，而**读**不改变任何状态。

先把单层的 SAE 特征跑通（ReLU 之后的零比例就是"稀疏度"）：

In [2]:
DICT = 64  # SAE 字典大小


class SingleLayerSAE(pllm.Workload):
    encoder = pllm.InstanceState(
        {"w3": lambda model: (DICT, model.hidden_size)},
        dtype=torch.float32,
        initialize=lambda tensors, generator: [
            torch.nn.init.orthogonal_(tensor, generator=generator)
            for tensor in tensors.values()],
    )

    # max_cols 同时界定输入帧宽与导出列数：输入要整宽（896），输出只有 64 列
    @pllm.on(pllm.Phase.OBSERVE)
    @pllm.export(name="sae3", every=1, max_rows=4, max_cols=896)
    def encode(self, h3: Annotated[torch.Tensor, pllm.Hidden(3)],
               w3: Annotated[torch.Tensor, pllm.Parameter("encoder", "w3")]) -> torch.Tensor:
        return torch.relu(h3.float() @ w3.T)

## 第三步：三层一起观测 —— 多输入任务

一个任务可以声明多个 `Hidden` 输入和多个 `Parameter` 输入。我们把三层的
特征沿列拼接成一个 `(行数, 3 × DICT)` 的张量导出，宿主机按列块切回三层。

> 为什么是一个任务而不是三个？观测附件的签名约定是"一个观测任务"，
> 多读几路输入没有任何额外代价。

In [3]:
LAYERS = (3, 5, 17)


class SAEObserver(pllm.Workload):
    """3/5/17 层共享一个冻结编码器状态，逐步导出拼接特征。"""

    encoder = pllm.InstanceState(
        {f"w{layer}": lambda model: (DICT, model.hidden_size) for layer in LAYERS},
        dtype=torch.float32,
        initialize=lambda tensors, generator: [
            torch.nn.init.orthogonal_(tensor, generator=generator)
            for tensor in tensors.values()],
    )

    @pllm.on(pllm.Phase.OBSERVE)
    @pllm.export(name="sae", every=1, max_rows=4, max_cols=896)
    def encode(
        self,
        h3: Annotated[torch.Tensor, pllm.Hidden(3)],
        h5: Annotated[torch.Tensor, pllm.Hidden(5)],
        h17: Annotated[torch.Tensor, pllm.Hidden(17)],
        w3: Annotated[torch.Tensor, pllm.Parameter("encoder", "w3")],
        w5: Annotated[torch.Tensor, pllm.Parameter("encoder", "w5")],
        w17: Annotated[torch.Tensor, pllm.Parameter("encoder", "w17")],
    ) -> torch.Tensor:
        features = [torch.relu(h.float() @ w.T)
                    for h, w in ((h3, w3), (h5, w5), (h17, w17))]
        return torch.cat(features, dim=-1)

## 第四步：生成的同时打印与落盘 —— 并行消费 trace

trace 是有界、加锁的缓冲（oldest-drop），所以一个后台线程可以在生成进行的
同时轮询它。游标写法只有一行要点：事件的绝对序号 = `dropped + len(events)`，
每次从 `cursor - dropped` 切起即可不遗漏（前提是轮询比容量快）。

In [4]:
import gc
import pickle
import threading
import time

llm.close()   # 第一步的引擎退场
del llm
gc.collect()
torch.cuda.empty_cache()

llm = pllm.make_llm(workload=SAEObserver(), config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", VllmDType.BFLOAT16, 64, 2, 128, 0.2, seed=7))

drained = []
cursor = 0
done = threading.Event()

def drain():
    global cursor
    while not done.is_set():
        snapshot = llm.trace_snapshot()
        drained.extend(snapshot.events[max(0, cursor - snapshot.dropped):])
        cursor = snapshot.dropped + len(snapshot.events)
        time.sleep(0.01)

thread = threading.Thread(target=drain)
thread.start()
try:
    outputs = llm.generate(
        ["The sky is", "Water freezes at"],
        SamplingParams(max_tokens=8, temperature=0.0, ignore_eos=True),
        use_tqdm=False)
finally:
    done.set()
    thread.join()

# 收尾再取一次，防止与最后的事件竞争
snapshot = llm.trace_snapshot()
drained.extend(snapshot.events[max(0, cursor - snapshot.dropped):])

rows = []
for event in drained:
    if event.tensor_name != "sae":
        continue
    features = event.tensor()
    rows.append(features)
    sparsity = [(block == 0).float().mean().item()
                for block in features.split(DICT, dim=-1)]
    print(f"step {event.step_id}: 行数={features.shape[0]}, "
          f"各层稀疏度 {[f'{v:.2f}' for v in sparsity]}")

with open("sae_features.pkl", "wb") as handle:
    pickle.dump({"layers": LAYERS, "dict": DICT, "features": torch.cat(rows)}, handle)
print(f"已写入 sae_features.pkl: {tuple(torch.cat(rows).shape)}")

INFO 09-10 12:28:34 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 128, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 12:28:34 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 12:28:35 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 12:28:35 [model.py:1883] Using max model len 64


INFO 09-10 12:28:35 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 12:28:36 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=64, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (9 outputs left out)

INFO 09-10 12:28:38 [kv_cache_utils.py:2236] Maximum concurrency for 64 tokens per request: 5073.25x


INFO 09-10 12:28:38 [gpu_worker.py:789] Free memory on device (11.64/23.48 GiB) on startup. Desired GPU memory utilization is (0.2, 4.7 GiB). Actual usage is 0.98 GiB for consumed memory (weights + non-torch), 0.0 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=3832543232` (3.57 GiB) to fit into requested memory, or `--kv-cache-memory=11282675712` (10.51 GiB) to fully utilize gpu memory. Current kv cache memory in use is 3.72 GiB.


INFO 09-10 12:28:39 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.88 s


step 1: 行数=2, 各层稀疏度 ['0.48', '0.42', '0.53']
step 2: 行数=2, 各层稀疏度 ['0.47', '0.45', '0.47']
step 3: 行数=2, 各层稀疏度 ['0.45', '0.45', '0.47']
step 4: 行数=2, 各层稀疏度 ['0.41', '0.48', '0.48']
step 5: 行数=2, 各层稀疏度 ['0.45', '0.46', '0.48']
step 6: 行数=2, 各层稀疏度 ['0.49', '0.55', '0.45']
step 7: 行数=2, 各层稀疏度 ['0.55', '0.52', '0.52']
已写入 sae_features.pkl: (14, 192)


## 收尾检查

In [5]:
diagnostics = llm.worker_state().diagnostics
assert diagnostics.fallback_count == 0
assert diagnostics.pending_state_releases == 0
llm.close()
for output in outputs:
    print(output.outputs[0].text)   # 与无观测时完全一致：观测不改变行为

 a beautiful place. It is a place
 0 degrees Celsius and boils at 


## 要点回顾

- 观测的第一原则：**不许改变模型状态**；读冻结状态天然满足它。
- 冻结状态 = 无优化器的 `InstanceState`：admission 初始化、处处可读、从不更新。
- `max_cols` 同时界定输入帧宽与导出列数；`max_rows × max_cols × 8 ≤ 32 KiB`。
- trace 有界、加锁、oldest-drop：后台线程用 `dropped + len(events)` 游标并行消费。

下一篇：[教程 2：写自己的训练头](02_custom_head.ipynb) —— 从一行 loss 到 CNN。